In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm'], check=False)

In [ ]:
import os, glob, json, time, math, copy, gc, platform, tarfile, tracemalloc, urllib.request, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings('ignore')

CFG = dict(
    ALIBABA_DIRS=['/kaggle/input/datasets/rohitdurbha/alibaba-cluster-data', '/kaggle/input/datasets/tirkeydivyanshu/alibaba-cluster-traces-2017'],
    KAGGLE_DATASET='lakshitha323/alibaba-cluster-trace-2018-part1',   # only the batch_task file is fetched via the Kaggle API (not attached)
    BATCH_TASK_URL='http://clusterdata2018pubcn.oss-cn-beijing.aliyuncs.com/batch_task.tar.gz',   # official fallback
    SYNTH_CSV='/kaggle/input/datasets/ziya07/cloud-task-scheduling-dataset/cloud_task_scheduling_dataset_20k.csv',
    OUT='/kaggle/working/hdmhcs_revision',
    SEED=42, N_SEEDS=10, MAX_TASKS=400_000,                # tasks kept from the trace (chronological)
    N_TASKS=1000, N_VMS=20,                               # main experiment
    POP=40, ITERS=100,                                    # identical for GA, PSO, HHO, HD-MHCS → 4,000 fitness evaluations
    WEIGHTS=(0.25, 0.25, 0.25, 0.25),                     # makespan, energy, cost, (1 − utilization)
    DEADLINE_FACTOR=1.0, REF_MEM_GB=64.0,
    DL_EPOCHS=30, DL_BATCH=512, DL_PATIENCE=5,
    DQN_EPISODES=300, DQN_TASKS=200, DQN_GAMMA=0.99, DQN_BUFFER=50_000, DQN_TARGET_SYNC=500, DQN_BATCH=128,
    SCALE_TASKS=[500, 1000, 2000, 5000, 10000], SCALE_VMS=[10, 20, 50], SCALE_SEEDS=3,
    WEIGHT_GRID=[(1, 0, 0, 0), (0.75, 0.25, 0, 0), (0.5, 0.5, 0, 0), (0.25, 0.75, 0, 0), (0, 1, 0, 0), (0.25, 0.25, 0.25, 0.25)],
    QUICK=False,
)
if os.environ.get('H_DIR'): CFG['ALIBABA_DIRS'] = [os.environ['H_DIR']]
if os.environ.get('H_OUT'): CFG['OUT'] = os.environ['H_OUT']
if os.environ.get('H_SYNTH'): CFG['SYNTH_CSV'] = os.environ['H_SYNTH']
if os.environ.get('H_QUICK') == '1': CFG['QUICK'] = True
if CFG['QUICK']:
    CFG.update(N_SEEDS=3, N_TASKS=300, N_VMS=10, ITERS=15, MAX_TASKS=60_000, DL_EPOCHS=2, DQN_EPISODES=10, DQN_TASKS=50,
               SCALE_TASKS=[300, 600], SCALE_VMS=[10], SCALE_SEEDS=1, WEIGHT_GRID=[(1, 0, 0, 0), (0, 1, 0, 0)])
OUT = CFG['OUT']
for s in ['tables', 'figures']: os.makedirs(os.path.join(OUT, s), exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
plt.rcParams.update({'font.size': 14, 'axes.labelsize': 15, 'legend.fontsize': 12, 'savefig.dpi': 300})
TABLES = {}
def save_table(df, name, show=True):
    TABLES[name] = df.copy(); df.to_csv(os.path.join(OUT, 'tables', name + '.csv'), index=False)
    if show: print(f'\n=== {name} ===\n' + (df.to_string(index=False) if len(df) <= 60 else df.head(60).to_string(index=False)))
    return df
def save_fig(fig, name): fig.tight_layout(); fig.savefig(os.path.join(OUT, 'figures', name + '.png'), bbox_inches='tight'); plt.close(fig)
def set_seed(s): np.random.seed(s); torch.manual_seed(s)
def sh(c):
    try: return subprocess.run(c, shell=True, capture_output=True, text=True, timeout=20).stdout.strip()
    except Exception: return ''
env = dict(python=platform.python_version(), torch=torch.__version__, numpy=np.__version__,
           gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
           cpu=(sh("grep -m1 'model name' /proc/cpuinfo | cut -d: -f2") or platform.processor()).strip(), cpu_cores=os.cpu_count(),
           ram_GB=round(int(sh("grep MemTotal /proc/meminfo | awk '{print $2}'") or 0) / 1e6, 1))
save_table(pd.DataFrame(list(env.items()), columns=['item', 'value']), 'T_environment')

In [ ]:
if not os.path.exists(CFG['SYNTH_CSV']):
    _h = sorted(glob.glob('/kaggle/input/**/*cloud*task*sched*.csv', recursive=True))
    if _h: CFG['SYNTH_CSV'] = _h[0]
if os.path.exists(CFG['SYNTH_CSV']):
    sd = pd.read_csv(CFG['SYNTH_CSV']); tgt = [c for c in sd.columns if 'Target' in c][0]
    feats = [c for c in sd.columns if c not in (tgt, 'Task_ID')]
    from sklearn.model_selection import train_test_split
    a, b = train_test_split(np.arange(len(sd)), test_size=0.3, random_state=0, stratify=sd[tgt])
    dt = DecisionTreeClassifier(max_depth=3, random_state=0).fit(sd.loc[a, feats], sd.loc[a, tgt])
    acc = (dt.predict(sd.loc[b, feats]) == sd.loc[b, tgt]).mean()
    save_table(pd.DataFrame([dict(rows=len(sd), class_1_share=sd[tgt].mean(), depth3_tree_test_accuracy=acc)]), 'T_synthetic_label_diagnosis')
    rule = export_text(dt, feature_names=feats); print(rule); open(os.path.join(OUT, 'tables', 'synthetic_label_tree.txt'), 'w').write(rule)
else:
    print('synthetic dataset not attached — diagnosis skipped')

## Alibaba Cluster Trace 2018 — `batch_task` (real production tasks)
Columns: task_name, instance_num, job_name, task_type, status, start_time, end_time, plan_cpu (100 = 1 core),
plan_mem (% of a machine). Kept: terminated tasks with valid times and requests. Runtime = end − start (s).
CPU work W = runtime × plan_cpu/100 (core-seconds); memory = plan_mem/100 × 64 GB.

In [ ]:
BT_COLS = ['task_name', 'instance_num', 'job_name', 'task_type', 'status', 'start_time', 'end_time', 'plan_cpu', 'plan_mem']
def find_batch_task():
    files = [f for d in CFG['ALIBABA_DIRS'] if os.path.isdir(d) for f in glob.glob(os.path.join(d, '**', '*'), recursive=True) if os.path.isfile(f)]
    files += [f for f in glob.glob('/kaggle/input/**/*batch_task*', recursive=True) if os.path.isfile(f)]
    save_table(pd.DataFrame(dict(file=sorted(set(files)), size_MB=[round(os.path.getsize(f) / 1e6, 1) for f in sorted(set(files))])), 'T_input_files')
    hits = [f for f in files if 'batch_task' in os.path.basename(f).lower()]
    def nfields(f):
        try:
            if f.endswith('.csv'):
                with open(f) as fh: return len(fh.readline().strip().split(','))
        except Exception: pass
        return 0
    if hits:
        pick = sorted(hits, key=lambda f: (nfields(f) != 9, not f.endswith('.csv'), f))[0]
        print('batch_task file used:', pick, '| fields per line:', nfields(pick)); return pick
    # ---- (1) Kaggle API: download ONLY the batch_task file from the large dataset (no mounting of 116 GB) ----
    try:
        try:
            from kaggle_secrets import UserSecretsClient
            sc_ = UserSecretsClient(); os.environ['KAGGLE_USERNAME'] = sc_.get_secret('KAGGLE_USERNAME'); os.environ['KAGGLE_KEY'] = sc_.get_secret('KAGGLE_KEY')
        except Exception as e:
            print('  (Kaggle secrets KAGGLE_USERNAME / KAGGLE_KEY not set:', e, ')')
        if os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'):
            from kaggle.api.kaggle_api_extended import KaggleApi
            api = KaggleApi(); api.authenticate(); ds = CFG['KAGGLE_DATASET']; names = []; tok = None
            while True:
                res = api.dataset_list_files(ds, page_token=tok) if 'page_token' in api.dataset_list_files.__code__.co_varnames else api.dataset_list_files(ds)
                names += [f.name for f in res.files]; tok = getattr(res, 'nextPageToken', None) or getattr(res, 'next_page_token', None)
                if not tok: break
            hit = sorted([n for n in names if 'batch_task' in n.lower()], key=lambda n: (not n.endswith('.csv'), n))
            print(f'  Kaggle API: {len(names)} files in {ds}; batch_task candidates: {hit[:5]}', flush=True)
            if hit:
                ddir = '/kaggle/working/alibaba' if os.path.isdir('/kaggle/working') else os.path.join(OUT, 'alibaba'); os.makedirs(ddir, exist_ok=True)
                t0 = time.time(); api.dataset_download_file(ds, hit[0], path=ddir, quiet=False)
                for z in glob.glob(os.path.join(ddir, '*.zip')):
                    import zipfile; zipfile.ZipFile(z).extractall(ddir); os.remove(z)
                got = [f for f in glob.glob(os.path.join(ddir, '**', '*'), recursive=True) if 'batch_task' in os.path.basename(f).lower() and os.path.isfile(f)]
                if got:
                    print(f'  downloaded {os.path.basename(got[0])} via the Kaggle API in {(time.time()-t0)/60:.1f} min', flush=True); return got[0]
    except Exception as e:
        print('  Kaggle API download failed:', e, flush=True)
    # ---- (2) official Alibaba server ----
    dst = '/kaggle/working/batch_task.tar.gz' if os.path.isdir('/kaggle/working') else os.path.join(OUT, 'batch_task.tar.gz')
    print('batch_task not found in the inputs → downloading the official file (~125 MB) …', flush=True)
    t0, got, last = time.time(), 0, 0
    try:
        with urllib.request.urlopen(CFG['BATCH_TASK_URL'], timeout=60) as r, open(dst, 'wb') as fh:
            total = int(r.headers.get('Content-Length') or 0)
            while True:
                chunk = r.read(1 << 20)
                if not chunk: break
                fh.write(chunk); got += len(chunk)
                if got - last >= 10 << 20:
                    last = got; rate = got / 1e6 / max(time.time() - t0, 1e-9)
                    print(f'  {got/1e6:6.0f} MB' + (f' of {total/1e6:.0f} MB' if total else '') + f'  ({rate:.2f} MB/s)', flush=True)
                if time.time() - t0 > 1200 and got < 30e6:
                    raise TimeoutError('download too slow (< 30 MB in 20 min)')
    except Exception as e:
        raise SystemExit(f'\n!! Could not download batch_task ({e}).\n!! Download batch_task.tar.gz on your computer (GitHub alibaba/clusterdata → '
                         f'cluster-trace-v2018), upload it to Kaggle as a new dataset, attach it, and run again.')
    print(f'  download finished: {got/1e6:.0f} MB in {(time.time()-t0)/60:.1f} min', flush=True); return dst

def read_batch_task(path):
    """fast single-pass read with PyArrow (multithreaded, fixed column types); falls back to pandas chunks"""
    t0 = time.time()
    if path.endswith(('.tar.gz', '.tgz')):
        tf = tarfile.open(path); m = [x for x in tf.getmembers() if x.name.endswith('.csv')][0]
        dst = '/kaggle/working' if os.path.isdir('/kaggle/working') else OUT
        tf.extract(m, dst); path = os.path.join(dst, m.name); print('  extracted', path)
    with open(path) as fh: first = fh.readline().strip().split(',')
    header = any(h in ','.join(first).lower() for h in ('plan_cpu', 'plan_mem', 'status', 'instance_num', 'timestamp'))
    if len(first) == 8:                                   # Alibaba v2017 batch_task
        cols17 = ['start_time', 'end_time', 'job_name', 'task_name', 'instance_num', 'status', 'plan_cpu', 'plan_mem']
        df = pd.read_csv(path, header=0 if header else None, names=cols17)
        df['task_type'] = '1'; n_read = len(df); CFG['TRACE_VERSION'] = 'Alibaba cluster trace v2017'
        for c in ['start_time', 'end_time', 'instance_num', 'plan_cpu', 'plan_mem']: df[c] = pd.to_numeric(df[c], errors='coerce')
        df['task_name'] = df['task_name'].astype(str); df['job_name'] = df['job_name'].astype(str)
        if df.plan_mem.max() <= 1.5: df['plan_mem'] = df.plan_mem * 100        # normalized (0–1) → percent
        if df.plan_cpu.max() <= 64: df['plan_cpu'] = df.plan_cpu * 100         # cores → 100 = 1 core
        df = df[(df.status.astype(str).str.lower() == 'terminated') & (df.end_time > df.start_time) & (df.start_time > 0) & (df.plan_cpu > 0) & (df.plan_mem > 0)]
        print(f'  v2017 format: read {n_read:,} rows in {time.time()-t0:.0f} s'); return df[BT_COLS].reset_index(drop=True), n_read
    CFG['TRACE_VERSION'] = 'Alibaba cluster trace v2018'
    if header: print('  header line detected and skipped')
    try:
        import pyarrow as pa, pyarrow.csv as pcsv
        types = {'task_name': pa.string(), 'instance_num': pa.float64(), 'job_name': pa.string(), 'task_type': pa.string(), 'status': pa.string(),
                 'start_time': pa.float64(), 'end_time': pa.float64(), 'plan_cpu': pa.float64(), 'plan_mem': pa.float64()}
        tbl = pcsv.read_csv(path, read_options=pcsv.ReadOptions(column_names=BT_COLS, block_size=64 << 20, use_threads=True, skip_rows=1 if header else 0),
                            convert_options=pcsv.ConvertOptions(column_types=types))
        df = tbl.to_pandas(); n_read = len(df)
    except Exception as e:
        print('  PyArrow reader failed (', e, ') → pandas chunks')
        df = pd.concat(pd.read_csv(path, header=None, names=BT_COLS, chunksize=2_000_000), ignore_index=True); n_read = len(df)
    df = df[(df.status == 'Terminated') & (df.end_time > df.start_time) & (df.start_time > 0) & (df.plan_cpu > 0) & (df.plan_mem > 0)]
    print(f'  read {n_read:,} rows in {time.time()-t0:.0f} s')
    return df.reset_index(drop=True), n_read

import hashlib
CACHE_NAME = f'alibaba_tasks_cache_{CFG["MAX_TASKS"]}_{hashlib.md5(str(sorted(CFG["ALIBABA_DIRS"])).encode()).hexdigest()[:8]}.parquet'
_cache = sorted(glob.glob(os.path.join('/kaggle/input', '**', CACHE_NAME), recursive=True)) + \
         [p for p in [os.path.join(OUT, CACHE_NAME)] if os.path.exists(p)]
if _cache:
    t0 = time.time(); BT = pd.read_parquet(_cache[0]); meta = json.load(open(_cache[0] + '.json')) if os.path.exists(_cache[0] + '.json') else {}
    N_READ = meta.get('rows_read', len(BT)); BT_PATH = meta.get('source', 'cache')
    FEATS = meta.get('feats') or ['instance_num', 'plan_cpu', 'plan_mem', 'task_type', 'job_n_tasks', 'job_total_inst', 'task_idx_in_job', 'is_dag_child',
                                  'n_parents', 'hour', 'hist_mean_logrt', 'hist_count', 'has_hist']
    print(f'Loaded cached task table {_cache[0]} ({len(BT):,} tasks) in {time.time()-t0:.1f} s — trace reading skipped')
else:
    BT_PATH = find_batch_task(); BT, N_READ = read_batch_task(BT_PATH)
    BT = BT.sort_values(['start_time', 'job_name', 'task_name']).reset_index(drop=True)
    BT['runtime'] = (BT.end_time - BT.start_time).astype(float)
    BT = BT[(BT.runtime >= 1) & (BT.runtime <= BT.runtime.quantile(0.999))]
    BT = BT.iloc[:CFG['MAX_TASKS']].reset_index(drop=True)
    BT['cores'] = BT.plan_cpu / 100.0; BT['work'] = BT.runtime * BT.cores; BT['mem_gb'] = BT.plan_mem / 100.0 * CFG['REF_MEM_GB']
    # submission-time features only (no runtime information)
    jg = BT.groupby('job_name')
    BT['job_n_tasks'] = jg.task_name.transform('count'); BT['job_total_inst'] = jg.instance_num.transform('sum')
    BT['task_idx_in_job'] = jg.cumcount(); BT['is_dag_child'] = BT.task_name.astype(str).str.contains('_').astype(int)
    BT['n_parents'] = BT.task_name.astype(str).str.count('_'); BT['hour'] = (BT.start_time // 3600) % 24
    BT['task_type'] = pd.to_numeric(BT.task_type, errors='coerce').fillna(0)
    # job history known at submission: runtimes of earlier tasks of the same job that had ALREADY FINISHED (end < start)
    def job_history(df):
        lr = np.log1p(df.runtime.values); hm = np.full(len(df), np.nan); hc = np.zeros(len(df))
        for _, ix in df.groupby('job_name').indices.items():
            if len(ix) < 2: continue
            st, en, l = df.start_time.values[ix], df.end_time.values[ix], lr[ix]
            for p, i in enumerate(ix):
                done = en < st[p]
                if done.any(): hm[i] = l[done].mean(); hc[i] = done.sum()
        return hm, hc
    t_h = time.time(); hm, hc = job_history(BT); print(f'job-history features in {time.time()-t_h:.0f} s; tasks with history: {np.mean(hc > 0):.1%}')
    BT['hist_count'] = hc; BT['has_hist'] = (hc > 0).astype(int)
    BT['hist_mean_logrt'] = np.where(hc > 0, hm, np.nanmean(np.log1p(BT.runtime.values[:int(.6 * len(BT))])))   # fallback: training-period mean
    FEATS = ['instance_num', 'plan_cpu', 'plan_mem', 'task_type', 'job_n_tasks', 'job_total_inst', 'task_idx_in_job', 'is_dag_child', 'n_parents', 'hour',
             'hist_mean_logrt', 'hist_count', 'has_hist']
    BT.to_parquet(os.path.join(OUT, CACHE_NAME), index=False)
    json.dump(dict(rows_read=int(N_READ), source=os.path.basename(BT_PATH), feats=FEATS), open(os.path.join(OUT, CACHE_NAME) + '.json', 'w'))
    print('Saved task-table cache →', os.path.join(OUT, CACHE_NAME), '(attach this notebook\'s output as an input next time)')
save_table(pd.DataFrame([dict(trace=CFG.get('TRACE_VERSION', 'Alibaba cluster trace'), source=os.path.basename(BT_PATH), rows_read=N_READ, tasks_kept=len(BT), runtime_median_s=BT.runtime.median(),
           runtime_p95_s=BT.runtime.quantile(.95), plan_cpu_median=BT.plan_cpu.median(), plan_mem_median=BT.plan_mem.median(),
           time_span_h=(BT.start_time.max() - BT.start_time.min()) / 3600)]), 'T_trace_summary')

# chronological split (no leakage): first 60% train, next 20% validation, last 20% test (by submission time)
n = len(BT); TR, VA, TE = np.arange(0, int(.6 * n)), np.arange(int(.6 * n), int(.8 * n)), np.arange(int(.8 * n), n)
SC = StandardScaler().fit(BT.loc[TR, FEATS].values)                     # fitted on the training period only
XF = SC.transform(BT[FEATS].values).astype(np.float32); YL = np.log1p(BT.runtime.values).astype(np.float32)

In [ ]:
VM_TYPES = pd.DataFrame(dict(type=['small', 'medium', 'large', 'xlarge'], cores=[2, 4, 8, 16], speed=[1.0, 1.1, 1.2, 1.3],
                             ram_gb=[8, 16, 32, 64], p_busy=[60, 110, 200, 380], p_idle=[30, 50, 90, 160], price_h=[0.05, 0.10, 0.20, 0.40]))
save_table(VM_TYPES, 'T_vm_types')

def make_vms(M, seed=0):
    rng = np.random.default_rng(seed); t = np.resize(np.arange(4), M); rng.shuffle(t); v = VM_TYPES.iloc[t].reset_index(drop=True)
    return dict(rate=(v.cores * v.speed).values.astype(float), ram=v.ram_gb.values.astype(float), pb=v.p_busy.values.astype(float),
                pi=v.p_idle.values.astype(float), price=v.price_h.values.astype(float), type=v.type.values)

class Batch:
    """a batch of N tasks to schedule on M VMs; work_true for evaluation, work_used for the scheduler (true/predicted)"""
    def __init__(self, idx, M, work_used=None, seed=0):
        self.idx = idx; self.N = len(idx); self.M = M; self.vm = make_vms(M, seed)
        self.work_true = BT.work.values[idx]; self.work_used = self.work_true if work_used is None else work_used
        self.mem = BT.mem_gb.values[idx]
        rref = np.median(self.vm['rate']); tref = self.work_true / rref
        self.deadline = CFG['DEADLINE_FACTOR'] * (self.N / M) * tref.mean() + tref
        self.feasible = self.vm['ram'][None, :] >= self.mem[:, None]          # N × M
        order = np.argsort(self.vm['ram'], kind='stable'); self.ram_order = order
        self.ref = None; self.evals = 0; self.budget = float('inf')
        self.ref = self.raw(round_robin(self), self.work_used)
    def repair(self, a):
        bad = ~self.feasible[np.arange(self.N), a]
        if bad.any():
            for i in np.flatnonzero(bad):
                ok = np.flatnonzero(self.feasible[i]); a[i] = ok[np.argmin(np.abs(ok - a[i]))]
        return a
    def raw(self, a, work):
        t = work / self.vm['rate'][a]; B = np.bincount(a, weights=t, minlength=self.M); mk = B.max()
        o = np.lexsort((self.deadline, a)); tt = t[o]; aa = a[o]
        cs = np.cumsum(tt); starts = np.r_[0, np.flatnonzero(aa[1:] != aa[:-1]) + 1]
        base = np.repeat(np.r_[0, cs[starts[1:] - 1]] if len(starts) > 1 else np.array([0.0]), np.diff(np.r_[starts, len(aa)]))
        comp = np.empty(self.N); comp[o] = cs - base
        energy = (self.vm['pb'] * B + self.vm['pi'] * (mk - B)).sum() / 3600; cost = (self.vm['price'] * B).sum() / 3600
        return dict(makespan=mk, energy=energy, cost=cost, util=B.mean() / mk, sla=(comp > self.deadline).mean(), completed=self.N)
    def terms(self, a, work):
        """fast objective terms (makespan, energy, cost, utilization) — SLA is reported but not part of the fitness"""
        B = np.bincount(a, weights=work / self.vm['rate'][a], minlength=self.M); mk = B.max()
        return mk, (self.vm['pb'] * B + self.vm['pi'] * (mk - B)).sum() / 3600, (self.vm['price'] * B).sum() / 3600, B.mean() / mk
    def _fit(self, a, work, f, w):
        mk, en, co, ut = self.terms(a, work)
        return w[0] * mk / f['makespan'] + w[1] * en / f['energy'] + w[2] * co / f['cost'] + w[3] * (1 - ut) / max(1 - f['util'], 1e-9)
    def fitness(self, a, w=None):
        """one evaluation; with runtime scenarios (robust mode) the fitness is the mean over the K sampled scenarios"""
        self.evals += 1; w = w or CFG['WEIGHTS']
        if getattr(self, 'scen', None) is not None:
            return float(np.mean([self._fit(a, wk, fk, w) for wk, fk in zip(self.scen, self.scen_ref)]))
        return self._fit(a, self.work_used, self.ref, w)
    def evaluate(self, a): return self.raw(a, self.work_true)                 # true runtimes
    def exhausted(self): return self.evals >= self.budget

def round_robin(b): return b.repair(np.arange(b.N) % b.M)
def random_sched(b, rng): return b.repair(rng.integers(0, b.M, b.N))
def min_min(b, work):
    """Min-Min (MCT order by shortest task): each task to the VM with the earliest completion time"""
    ready = np.zeros(b.M); a = np.empty(b.N, int)
    for i in np.argsort(work):
        ct = ready + work[i] / b.vm['rate']; ct[~b.feasible[i]] = np.inf; j = int(ct.argmin()); a[i] = j; ready[j] = ct[j]
    return a
def max_min(b, work):
    ready = np.zeros(b.M); a = np.empty(b.N, int)
    for i in np.argsort(-work):
        ct = ready + work[i] / b.vm['rate']; ct[~b.feasible[i]] = np.inf; j = int(ct.argmin()); a[i] = j; ready[j] = ct[j]
    return a

In [ ]:
class BiLSTMAttn(nn.Module):
    def __init__(s, n, h=64):
        super().__init__(); s.emb = nn.Linear(1, 16); s.lstm = nn.LSTM(16, h, 2, batch_first=True, bidirectional=True, dropout=0.2)
        s.att = nn.Linear(2 * h, 1); s.out = nn.Sequential(nn.Linear(2 * h, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(s, x):
        H, _ = s.lstm(s.emb(x.unsqueeze(-1))); a = torch.softmax(s.att(H), 1); return s.out((a * H).sum(1)).squeeze(-1)
class ResMLP(nn.Module):
    def __init__(s, n, h=256, blocks=4):
        super().__init__(); s.inp = nn.Linear(n, h)
        s.blocks = nn.ModuleList([nn.Sequential(nn.Linear(h, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(0.15), nn.Linear(h, h)) for _ in range(blocks)])
        s.out = nn.Linear(h, 1)
    def forward(s, x):
        h = s.inp(x)
        for b in s.blocks: h = F.gelu(h + b(h))
        return s.out(h).squeeze(-1)
class TabTransformer(nn.Module):
    def __init__(s, n, d=32):
        super().__init__(); s.tok = nn.Parameter(torch.randn(n, d) * .1); s.val = nn.Linear(1, d)
        s.enc = nn.TransformerEncoder(nn.TransformerEncoderLayer(d, 4, 64, 0.1, batch_first=True), 2); s.out = nn.Linear(d, 1)
    def forward(s, x): return s.out(s.enc(s.val(x.unsqueeze(-1)) + s.tok).mean(1)).squeeze(-1)

def fit_reg(model, seed=42):
    set_seed(seed); model.to(DEVICE); opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    Xt, Yt = torch.tensor(XF[TR]), torch.tensor(YL[TR]); best, state, bad = 1e18, None, 0; rng = np.random.default_rng(seed)
    for ep in range(CFG['DL_EPOCHS']):
        model.train(); p = rng.permutation(len(TR))
        for b in range(0, len(p), CFG['DL_BATCH']):
            j = p[b:b + CFG['DL_BATCH']]; x, y = Xt[j].to(DEVICE), Yt[j].to(DEVICE)
            loss = F.smooth_l1_loss(model(x), y); opt.zero_grad(); loss.backward(); opt.step()
        v = mean_absolute_error(YL[VA], pred_reg(model, XF[VA]))
        if v < best - 1e-5: best, state, bad = v, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= CFG['DL_PATIENCE']: break
    model.load_state_dict(state); return model
@torch.no_grad()
def pred_reg(model, X):
    model.eval(); return np.concatenate([model(torch.tensor(X[i:i + 8192]).to(DEVICE)).cpu().numpy() for i in range(0, len(X), 8192)])

t0 = time.time(); PRED, rows = {}, []
for name, mk in [('BiLSTM-attention', lambda: BiLSTMAttn(len(FEATS))), ('Deep residual MLP', lambda: ResMLP(len(FEATS))), ('Transformer', lambda: TabTransformer(len(FEATS)))]:
    t1 = time.time(); m = fit_reg(mk()); PRED[name] = pred_reg(m, XF); rows.append((name, time.time() - t1))
try:
    import lightgbm as lgb; t1 = time.time()
    g = lgb.LGBMRegressor(n_estimators=500, num_leaves=63, learning_rate=0.05, random_state=42, verbose=-1).fit(XF[TR], YL[TR])
    PRED['LightGBM'] = g.predict(XF); rows.append(('LightGBM', time.time() - t1))
except Exception as e: print('LightGBM skipped', e)
PRED['Mean predictor'] = np.full(len(YL), YL[TR].mean()); rows.append(('Mean predictor', 0.0))
_deep = [k for k in ['BiLSTM-attention', 'Deep residual MLP', 'Transformer'] if k in PRED]
_bd = min(_deep, key=lambda k: mean_absolute_error(YL[VA], PRED[k][VA]))
if 'LightGBM' in PRED:
    PRED['Hybrid (deep + LightGBM)'] = (PRED[_bd] + PRED['LightGBM']) / 2; rows.append(('Hybrid (deep + LightGBM)', 0.0))
res = []
for name, sec in rows:
    p = PRED[name]
    res.append(dict(model=name, MAE_log_validation=mean_absolute_error(YL[VA], p[VA]), MAE_log=mean_absolute_error(YL[TE], p[TE]), RMSE_log=math.sqrt(mean_squared_error(YL[TE], p[TE])), R2_log=r2_score(YL[TE], p[TE]),
                    median_abs_error_s=np.median(np.abs(np.expm1(p[TE]) - BT.runtime.values[TE])), train_seconds=sec))
DLR = save_table(pd.DataFrame(res), 'T_runtime_prediction')
BEST_DL = DLR[DLR.model != 'Mean predictor'].sort_values('MAE_log_validation').model.iloc[0]
print('runtime predictor used by HD-MHCS (selected on VALIDATION MAE; test set not used for selection):', BEST_DL)
save_table(pd.DataFrame([dict(selected_predictor=BEST_DL, criterion='lowest validation MAE (log runtime)')]), 'T_selected_predictor')
# prediction-error spread on the VALIDATION period (log space), per predictor
SIGMA = {k: float(np.std(YL[VA] - v[VA])) for k, v in PRED.items()}
save_table(pd.DataFrame([dict(predictor=k, residual_sd_log_validation=v) for k, v in SIGMA.items()]), 'T_prediction_error_spread')
Z_EST = {'median': 0.0, 'q75': 0.674}
def pred_work(idx, name=None, est='median'):
    """runtime estimate used by the schedulers: 'median' = exp(μ) − 1; 'mean' = exp(μ + σ²/2) − 1 (log-normal mean);
    'q75' = exp(μ + 0.674σ) − 1 (cautious upper estimate); σ = validation residual SD of the predictor"""
    name = name or BEST_DL; mu = PRED[name][idx]; sg = SIGMA[name]
    lg = mu + (sg * sg / 2 if est == 'mean' else Z_EST.get(est, 0.0) * sg)
    return np.expm1(lg).clip(1) * BT.cores.values[idx]
def sample_work(idx, K, name=None, seed=7):
    """K plausible runtime scenarios drawn from each task's predictive log-normal distribution"""
    name = name or BEST_DL; mu = PRED[name][idx]; sg = SIGMA[name]; rng = np.random.default_rng(seed)
    return [np.expm1(mu + sg * rng.standard_normal(len(idx))).clip(1) * BT.cores.values[idx] for _ in range(K)]

In [ ]:
def decode(b, x): return b.repair(np.minimum((x * b.M).astype(int), b.M - 1))
def encode(b, a, rng, noise=0.0): return np.clip((a + rng.random(b.N)) / b.M + noise * rng.standard_normal(b.N), 0, 0.999999)

def ga(b, rng, P=None, T=None, w=None):
    P, T = P or CFG['POP'], T or CFG['ITERS']; pop = rng.integers(0, b.M, (P, b.N)); pop = np.array([b.repair(p) for p in pop])
    fit = np.array([b.fitness(p, w) for p in pop])
    for _ in range(T - 1):
        if b.exhausted(): break
        el = pop[np.argsort(fit)[:max(2, P // 5)]]; kids = []
        for _ in range(P):
            p1, p2 = el[rng.integers(len(el))], el[rng.integers(len(el))]; c = rng.integers(1, b.N)
            ch = np.r_[p1[:c], p2[c:]]; mu = rng.random(b.N) < 0.02; ch[mu] = rng.integers(0, b.M, mu.sum()); kids.append(b.repair(ch))
        pop = np.array(kids); fit = np.array([b.fitness(p, w) for p in pop])
    return pop[fit.argmin()]

def pso(b, rng, P=None, T=None, w=None):
    P, T = P or CFG['POP'], T or CFG['ITERS']; x = rng.random((P, b.N)); v = rng.uniform(-.1, .1, (P, b.N))
    f = np.array([b.fitness(decode(b, xi), w) for xi in x]); pb, pf = x.copy(), f.copy(); g = pb[pf.argmin()].copy()
    for _ in range(T - 1):
        if b.exhausted(): break
        r1, r2 = rng.random((P, b.N)), rng.random((P, b.N)); v = 0.7 * v + 1.5 * r1 * (pb - x) + 1.5 * r2 * (g - x); x = np.clip(x + v, 0, 0.999999)
        f = np.array([b.fitness(decode(b, xi), w) for xi in x]); im = f < pf; pb[im], pf[im] = x[im], f[im]; g = pb[pf.argmin()].copy()
    return decode(b, g)

def levy(rng, n, beta=1.5):
    s = (math.gamma(1 + beta) * math.sin(math.pi * beta / 2) / (math.gamma((1 + beta) / 2) * beta * 2 ** ((beta - 1) / 2))) ** (1 / beta)
    return 0.01 * rng.standard_normal(n) * s / np.abs(rng.standard_normal(n)) ** (1 / beta)

def hho(b, rng, P=None, T=None, w=None, init=None, decay='linear', use_levy=True, perturb=0.0):
    """Harris Hawks Optimization (Heidari et al., 2019). decay: 'linear' (standard), 'nonlinear' (Eq. 5), 'chaotic' (logistic map)."""
    P, T = P or CFG['POP'], T or CFG['ITERS']; x = rng.random((P, b.N)) if init is None else init.copy()
    f = np.array([b.fitness(decode(b, xi), w) for xi in x]); best = x[f.argmin()].copy(); bf = f.min(); chaos = 0.7
    b.iters_done = 0
    for t in range(10 * T):
        if b.exhausted(): break
        b.iters_done += 1; t = min(t, T - 1)
        if decay == 'linear': Ef = 2 * (1 - t / T)
        elif decay == 'nonlinear': Ef = 2 * math.exp(-(t / T) / max(0.4 * (1 - t / T), 1e-3))       # exponential-linear schedule (Eq. 5, θ = 0.4)
        else: chaos = 4 * chaos * (1 - chaos); Ef = 2 * (1 - t / T) * chaos                            # chaotic (logistic map)
        xm = x.mean(0)
        for i in range(P):
            E = Ef * (2 * rng.random() - 1); q, r = rng.random(), rng.random()
            if abs(E) >= 1:
                if q >= 0.5: xr = x[rng.integers(P)]; x[i] = xr - rng.random() * np.abs(xr - 2 * rng.random() * x[i])
                else: x[i] = (best - xm) - rng.random() * rng.random(b.N)
            else:
                J = 2 * (1 - rng.random()); dx = best - x[i]
                if r >= 0.5: x[i] = dx - E * np.abs(J * best - x[i]) if abs(E) < 0.5 else dx - E * np.abs(J * best - x[i]) + x[i]
                else:
                    Y = best - E * np.abs(J * best - x[i]) if abs(E) >= 0.5 else best - E * np.abs(J * best - xm)
                    Z = Y + (rng.random(b.N) * levy(rng, b.N) if use_levy else 0)
                    x[i] = Y if b.fitness(decode(b, np.clip(Y, 0, .999999)), w) < b.fitness(decode(b, np.clip(Z, 0, .999999)), w) else Z
            if perturb > 0: x[i] = x[i] + perturb * rng.standard_normal(b.N) * (1 - t / T)
            x[i] = np.clip(x[i], 0, 0.999999)
        f = np.array([b.fitness(decode(b, xi), w) for xi in x])
        if f.min() < bf: bf, best = f.min(), x[f.argmin()].copy()
    return decode(b, best)

def energy_greedy(b, work, alpha=0.5):
    """energy-aware greedy: longest task first, VM minimizing α·completion/mean + (1−α)·task energy/mean"""
    ready = np.zeros(b.M); a = np.empty(b.N, int); mt = (work / b.vm['rate'].mean()).mean(); me = mt * b.vm['pb'].mean()
    for i in np.argsort(-work):
        t = work[i] / b.vm['rate']; sc = alpha * (ready + t) / mt + (1 - alpha) * t * b.vm['pb'] / me
        sc[~b.feasible[i]] = np.inf; j = int(sc.argmin()); a[i] = j; ready[j] += t[j]
    return a

def local_search(b, a, w=None, max_evals=None, cand=6):
    """memetic refinement: move one of the `cand` longest tasks of the most loaded VM to the VM that improves the
    fitness most; repeat until no improving move or the evaluation budget is exhausted"""
    a = a.copy(); best = b.fitness(a, w); stop = b.evals + (max_evals if max_evals is not None else float('inf'))
    while b.evals < min(stop, b.budget):
        t = b.work_used / b.vm['rate'][a]; B = np.bincount(a, weights=t, minlength=b.M); j = int(B.argmax())
        on = np.flatnonzero(a == j); on = on[np.argsort(-t[on])][:cand]; move = None
        for i in on:
            for k in np.flatnonzero(b.feasible[i]):
                if k == j or b.evals >= min(stop, b.budget): continue
                a[i] = k; f = b.fitness(a, w); a[i] = j
                if f < best - 1e-12: best, move = f, (i, k)
        if move is None: break
        a[move[0]] = move[1]
    return a

def hd_mhcs(b, rng, P=None, T=None, w=None, seed_frac=0.5, seed_noise=0.08, decay='nonlinear', use_levy=True, perturb=0.02,
            seed_work=None, multi_seed=True, ls_frac=0.3):
    """HD-MHCS: (1) population seeded (fraction seed_frac) around heuristic schedules computed with the DL-predicted
    runtimes — Min-Min, Max-Min and energy-aware greedy (multi_seed) or Min-Min only — with N(0, seed_noise²) noise;
    (2) adaptive HHO search (nonlinear energy decay, Lévy flights, Gaussian perturbation) using (1 − ls_frac) of the
    budget; (3) memetic local search on the best schedule with the remaining budget. Same total budget as all methods."""
    P = P or CFG['POP']; k = int(round(seed_frac * P)); x = rng.random((P, b.N)); wk = b.work_used if seed_work is None else seed_work
    if k > 0:
        seeds = [min_min(b, wk), max_min(b, wk), energy_greedy(b, wk)] if multi_seed else [min_min(b, wk)]
        for i in range(k): x[i] = encode(b, seeds[i % len(seeds)], rng, seed_noise if i >= len(seeds) else 0.0)
    total = b.budget; b.budget = b.evals + (1 - ls_frac) * (total - b.evals)
    a = hho(b, rng, P, T, w, init=x, decay=decay, use_levy=use_levy, perturb=perturb)
    b.budget = total
    if k > 0:                                             # the seeds themselves are candidates (no worse than the best heuristic)
        for sd in seeds:
            if b.fitness(sd, w) < b.fitness(a, w): a = sd.copy()
    return local_search(b, a, w) if ls_frac > 0 else a

In [ ]:
class DuelingQ(nn.Module):
    def __init__(s, d, M, h=128):
        super().__init__(); s.f = nn.Sequential(nn.Linear(d, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU()); s.V = nn.Linear(h, 1); s.A = nn.Linear(h, M)
    def forward(s, x): z = s.f(x); a = s.A(z); return s.V(z) + a - a.mean(1, keepdim=True)

def dqn_state(b, i, ready, wpred):
    sc = wpred.mean() / b.vm['rate'].mean() * b.N / b.M + 1e-9
    return np.r_[wpred[i] / wpred.mean(), b.mem[i] / 64, ready / sc, b.vm['rate'] / b.vm['rate'].max(), b.feasible[i].astype(float)].astype(np.float32)

def dqn_rollout(net, b, wpred, eps, rng, buf=None):
    ready = np.zeros(b.M); a = np.empty(b.N, int); mean_t = (wpred / b.vm['rate'].mean()).mean(); mean_e = mean_t * b.vm['pb'].mean()
    s = dqn_state(b, 0, ready, wpred)
    for i in range(b.N):
        feas = b.feasible[i]
        if rng.random() < eps: j = int(rng.choice(np.flatnonzero(feas)))
        else:
            with torch.no_grad(): qv = net(torch.tensor(s)[None].to(DEVICE))[0].cpu().numpy()
            qv[~feas] = -1e9; j = int(qv.argmax())
        old = ready.max(); t = wpred[i] / b.vm['rate'][j]; ready[j] += t; a[i] = j
        r = -(ready.max() - old) / mean_t - 0.1 * (t * b.vm['pb'][j]) / mean_e
        s2 = dqn_state(b, i + 1, ready, wpred) if i + 1 < b.N else np.zeros_like(s); done = i + 1 == b.N
        if buf is not None: buf.append((s, j, r, s2, done, b.feasible[i + 1] if not done else np.ones(b.M, bool)))
        s = s2
    return a

def train_dqn(M, seed=42):
    set_seed(seed); rng = np.random.default_rng(seed); d = 2 + 3 * M
    net, tgt = DuelingQ(d, M).to(DEVICE), DuelingQ(d, M).to(DEVICE); tgt.load_state_dict(net.state_dict())
    opt = torch.optim.Adam(net.parameters(), lr=5e-4); buf, step, log = [], 0, []
    for ep in range(CFG['DQN_EPISODES']):
        st = rng.integers(0, len(TR) - CFG['DQN_TASKS']); idx = TR[st:st + CFG['DQN_TASKS']]
        b = Batch(idx, M, seed=0); wp = pred_work(idx); eps = max(0.05, 1 - ep / (0.6 * CFG['DQN_EPISODES']))
        a = dqn_rollout(net, b, wp, eps, rng, buf); buf[:] = buf[-CFG['DQN_BUFFER']:]
        for _ in range(CFG['DQN_TASKS'] // 4):
            if len(buf) < CFG['DQN_BATCH']: break
            bt = [buf[k] for k in rng.integers(0, len(buf), CFG['DQN_BATCH'])]
            S = torch.tensor(np.stack([x[0] for x in bt])).to(DEVICE); A_ = torch.tensor([x[1] for x in bt]).to(DEVICE)
            R = torch.tensor([x[2] for x in bt], dtype=torch.float32).to(DEVICE); S2 = torch.tensor(np.stack([x[3] for x in bt])).to(DEVICE)
            D = torch.tensor([x[4] for x in bt], dtype=torch.float32).to(DEVICE); MK = torch.tensor(np.stack([x[5] for x in bt])).to(DEVICE)
            with torch.no_grad():
                q2 = net(S2).masked_fill(~MK, -1e9).argmax(1, keepdim=True)              # double-DQN action selection
                y = R + CFG['DQN_GAMMA'] * (1 - D) * tgt(S2).gather(1, q2).squeeze(1)
            loss = F.smooth_l1_loss(net(S).gather(1, A_[:, None]).squeeze(1), y); opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 10); opt.step(); step += 1
            if step % CFG['DQN_TARGET_SYNC'] == 0: tgt.load_state_dict(net.state_dict())
        log.append(dict(episode=ep + 1, epsilon=eps, makespan_true=b.evaluate(a)['makespan']))
    return net, pd.DataFrame(log)

t0 = time.time(); DQN_NET, DQN_LOG = train_dqn(CFG['N_VMS']); save_table(DQN_LOG.iloc[::max(1, len(DQN_LOG) // 20)], 'T_dqn_training_log')
print(f'DQN trained in {(time.time()-t0)/60:.1f} min')

In [ ]:
BUDGET = CFG['POP'] * CFG['ITERS']
TEST_IDX = TE[:CFG['N_TASKS']]
def new_batch(idx, M, oracle=False, work_name=None, est='median', K=0):
    b = Batch(idx, M, work_used=None if oracle else pred_work(idx, work_name, est), seed=0); b.scen = None
    if K and not oracle:
        b.scen = sample_work(idx, K, work_name); b.scen_ref = [b.raw(round_robin(b), wk) for wk in b.scen]
    return b

def run(name, b, rng, **kw):
    b.evals, b.budget, b.iters_done = 0, BUDGET, None; t0 = time.time(); tracemalloc.start()
    if name == 'Random': a = random_sched(b, rng)
    elif name == 'Round-robin': a = round_robin(b)
    elif name == 'Min-Min': a = min_min(b, b.work_used)
    elif name == 'Max-Min': a = max_min(b, b.work_used)
    elif name == 'GA': a = ga(b, rng, **kw)
    elif name == 'PSO': a = pso(b, rng, **kw)
    elif name == 'HHO': a = hho(b, rng, **kw)
    elif name == 'Dueling-DQN': a = dqn_rollout(DQN_NET, b, b.work_used, 0.0, rng)
    else: a = hd_mhcs(b, rng, **kw)
    peak = tracemalloc.get_traced_memory()[1] / 1e6; tracemalloc.stop()
    r = b.evaluate(a); r.update(method=name, wall_s=time.time() - t0, fitness_evals=b.evals, iterations=b.iters_done, peak_MB=peak)
    wt, ref, sc = b.work_used, b.ref, b.scen; b.work_used, b.scen = b.work_true, None; b.ref = b.raw(round_robin(b), b.work_true)
    r['fitness_true'] = b.fitness(a); b.work_used, b.ref, b.scen = wt, ref, sc       # fitness re-computed with the true runtimes
    return r

# ---- all choices on a VALIDATION-period batch (true runtimes of the validation period are known; test batch untouched) ----
VAL_IDX = VA[:CFG['N_TASKS']]; ESTS = ['median', 'mean', 'q75']
# (a) runtime estimate for EACH baseline: the estimate that is best for that method on validation (fair comparison)
rows, VBASE = [], {est: new_batch(VAL_IDX, CFG['N_VMS'], est=est) for est in ESTS}
for m in ['Min-Min', 'Max-Min', 'GA', 'PSO', 'HHO', 'Dueling-DQN']:
    for est in ESTS:
        nrep = 1 if m in ('Min-Min', 'Max-Min', 'Dueling-DQN') else 2
        f_ = np.mean([run(m, VBASE[est], np.random.default_rng(600 + k))['fitness_true'] for k in range(nrep)])
        rows.append(dict(method=m, estimate=est, val_fitness=f_))
ES = save_table(pd.DataFrame(rows), 'T_estimate_selection_baselines')
EST_BY = {m: g.sort_values('val_fitness').estimate.iloc[0] for m, g in ES.groupby('method')}
BASE_EST = EST_BY['Min-Min']; print('runtime estimate selected on validation for each baseline:', EST_BY)
# (b) HD-MHCS configuration: runtime estimate × robust scenario fitness × seeding × local search
grid, VB = [], {}
for est in ESTS:
    for K in [0, 8]:
        VB[(est, K)] = new_batch(VAL_IDX, CFG['N_VMS'], est=est, K=K)
        for ms_ in [True, False]:
            for lf in [0.0, 0.3]:
                fs = [run('HD-MHCS', VB[(est, K)], np.random.default_rng(500 + s), multi_seed=ms_, ls_frac=lf, seed_frac=0.5, perturb=0.02)['fitness_true'] for s in range(3)]
                grid.append(dict(estimate=est, robust_scenarios=K, multi_seed=ms_, ls_frac=lf, val_fitness_mean=np.mean(fs)))
TUNE = save_table(pd.DataFrame(grid).sort_values('val_fitness_mean'), 'T_hdmhcs_tuning_validation')
best = TUNE.iloc[0]; HD_EST, HD_K = best.estimate, int(best.robust_scenarios)
HD_KW = dict(multi_seed=bool(best.multi_seed), ls_frac=float(best.ls_frac), seed_frac=0.5, perturb=0.02)
print(f'HD-MHCS configuration selected on validation: estimate={HD_EST}, robust scenarios={HD_K}, {HD_KW}')
save_table(pd.DataFrame([dict(**{f'estimate_{k}': v for k, v in EST_BY.items()}, hd_estimate=HD_EST, hd_robust_scenarios=HD_K, **HD_KW)]), 'T_selected_configuration')
_run_orig = run
def run(name, b, rng, **kw):
    if name == 'HD-MHCS': kw = {**HD_KW, **kw}
    return _run_orig(name, b, rng, **kw)
def hd_batch(idx, M, **kw): return new_batch(idx, M, est=kw.pop('est', HD_EST), K=kw.pop('K', HD_K), **kw)
def base_batch(idx, M, method='Min-Min'): return new_batch(idx, M, est=EST_BY.get(method, 'median'))

METHODS = ['Random', 'Round-robin', 'Min-Min', 'Max-Min', 'GA', 'PSO', 'HHO', 'Dueling-DQN', 'HD-MHCS']
rows = []; b_hd = hd_batch(TEST_IDX, CFG['N_VMS'])
B_TEST = {est: new_batch(TEST_IDX, CFG['N_VMS'], est=est) for est in ESTS}           # each baseline uses its own selected estimate
for s in range(CFG['N_SEEDS']):
    for m in METHODS:
        b_ = b_hd if m == 'HD-MHCS' else B_TEST[EST_BY.get(m, 'median')]
        r = run(m, b_, np.random.default_rng(1000 + s)); r['seed'] = s; r['runtime_estimate'] = HD_EST if m == 'HD-MHCS' else EST_BY.get(m, 'median'); rows.append(r)
    bo = new_batch(TEST_IDX, CFG['N_VMS'], oracle=True)
    r = run('HD-MHCS', bo, np.random.default_rng(1000 + s)); r.update(seed=s, method='HD-MHCS (oracle runtimes)'); rows.append(r)
    print(f'  seed {s} done')
MAIN = save_table(pd.DataFrame(rows), 'T_main_runs', show=False)
def ci(x): x = np.asarray(x, float); return stats.t.ppf(.975, len(x) - 1) * x.std(ddof=1) / math.sqrt(len(x)) if len(x) > 1 else np.nan
KP = ['makespan', 'energy', 'cost', 'util', 'sla', 'completed', 'fitness_true', 'wall_s', 'fitness_evals']
summ = MAIN.groupby('method', sort=False).agg(**{f'{k}_mean': (k, 'mean') for k in KP}, **{f'{k}_sd': (k, 'std') for k in KP[:5]},
                                               fitness_ci95=('fitness_true', ci), makespan_ci95=('makespan', ci)).reset_index()
save_table(summ, 'T_main_summary')

def a12(x, y):  # Vargha–Delaney: probability that a value of x is smaller (better) than a value of y
    x, y = np.asarray(x), np.asarray(y); return ((x[:, None] < y[None, :]).sum() + 0.5 * (x[:, None] == y[None, :]).sum()) / (len(x) * len(y))
def holm(p):
    p = np.asarray(p, float); o = np.argsort(np.nan_to_num(p, nan=1)); out = np.full_like(p, np.nan); run_ = 0
    for r_, i in enumerate(o):
        if not np.isnan(p[i]): run_ = max(run_, min(1, (len(p) - r_) * p[i])); out[i] = run_
    return out
tests = []; hd = MAIN[MAIN.method == 'HD-MHCS'].sort_values('seed')
for m in [x for x in METHODS if x != 'HD-MHCS']:
    o = MAIN[MAIN.method == m].sort_values('seed')
    for k in ['fitness_true', 'makespan', 'energy', 'sla']:
        d = hd[k].values - o[k].values
        try: p = stats.wilcoxon(hd[k].values, o[k].values).pvalue if np.any(d != 0) else 1.0
        except Exception: p = np.nan
        tests.append(dict(comparison=f'HD-MHCS vs {m}', metric=k, mean_diff=d.mean(), p_wilcoxon=p, A12_HD_better=a12(hd[k].values, o[k].values)))
tests = pd.DataFrame(tests); tests['p_holm'] = holm(tests.p_wilcoxon.values); save_table(tests, 'T_statistical_tests')
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
for a_, k, lab in zip(ax, ['makespan', 'energy', 'sla'], ['Makespan (s)', 'Energy (Wh)', 'SLA violation rate']):
    g = MAIN.groupby('method', sort=False)[k]; a_.bar(range(g.ngroups), g.mean(), yerr=g.std(), capsize=3); a_.set_xticks(range(g.ngroups))
    a_.set_xticklabels(g.mean().index, rotation=45, ha='right'); a_.set_ylabel(lab)
save_fig(fig, 'F_main_comparison')

## Ablation (same task set, same budget, 10 seeds; scheduling metrics)

In [ ]:
ABL = {'HD-MHCS (full)': dict(), 'w/o local search': dict(ls_frac=0.0), 'Min-Min seed only (no multi-heuristic seeding)': dict(multi_seed=False),
       'w/o DL seeding (random init)': dict(seed_frac=0.0), 'seeded population only (100%)': dict(seed_frac=1.0),
       'w/o Gaussian perturbation': dict(seed_noise=0.0, perturb=0.0), 'linear energy decay (standard HHO)': dict(decay='linear'),
       'chaotic energy decay (logistic map)': dict(decay='chaotic'), 'w/o Lévy flight': dict(use_levy=False)}
b_abl_nok = hd_batch(TEST_IDX, CFG['N_VMS'], K=0); b_abl_med = hd_batch(TEST_IDX, CFG['N_VMS'], est='median')
ABL_SRC = [(f'seeding from {lbl}', hd_batch(TEST_IDX, CFG['N_VMS'], work_name=src)) for lbl, src in
           [('mean runtime (no prediction)', 'Mean predictor'), ('LightGBM only', 'LightGBM'), ('the best deep model only', _bd)] if src in PRED]
rows = []
for s in range(CFG['N_SEEDS']):
    for name, kw in ABL.items():
        r = run('HD-MHCS', b_hd, np.random.default_rng(2000 + s), **kw); r.update(variant=name, seed=s); rows.append(r)
    for name, bb in [('w/o robust scenario fitness', b_abl_nok), ('median runtime estimate', b_abl_med)] + [(n_, b_) for n_, b_ in ABL_SRC]:
        r = run('HD-MHCS', bb, np.random.default_rng(2000 + s)); r.update(variant=name, seed=s); rows.append(r)
AB = save_table(pd.DataFrame(rows), 'T_ablation_runs', show=False)
save_table(AB.groupby('variant', sort=False).agg(**{f'{k}_mean': (k, 'mean') for k in ['fitness_true', 'makespan', 'energy', 'cost', 'util', 'sla', 'completed']},
                                                  fitness_sd=('fitness_true', 'std')).reset_index(), 'T_ablation_summary')
full = AB[AB.variant == 'HD-MHCS (full)'].sort_values('seed'); t = []
for v in AB.variant.unique()[1:]:
    o = AB[AB.variant == v].sort_values('seed'); d = full.fitness_true.values - o.fitness_true.values
    try: p = stats.wilcoxon(full.fitness_true.values, o.fitness_true.values).pvalue if np.any(d != 0) else 1.0
    except Exception: p = np.nan
    t.append(dict(comparison=f'full vs {v}', mean_fitness_diff=d.mean(), p_wilcoxon=p, A12_full_better=a12(full.fitness_true.values, o.fitness_true.values)))
t = pd.DataFrame(t); t['p_holm'] = holm(t.p_wilcoxon.values); save_table(t, 'T_ablation_tests')

## Weight sensitivity and Pareto front (makespan vs energy)

In [ ]:
rows = []
for w in CFG['WEIGHT_GRID']:
    for s in range(3):
        r = run('HD-MHCS', b_hd, np.random.default_rng(3000 + s), w=w); r.update(weights=str(w), seed=s); rows.append(r)
WS = save_table(pd.DataFrame(rows).groupby('weights', sort=False)[['makespan', 'energy', 'cost', 'util', 'sla', 'completed']].mean().reset_index(), 'T_weight_sensitivity')
fig, ax = plt.subplots(figsize=(8, 6)); ax.plot(WS.makespan, WS.energy, 'o-')
for _, r in WS.iterrows(): ax.annotate(r.weights, (r.makespan, r.energy), fontsize=9)
ax.set_xlabel('Makespan (s)'); ax.set_ylabel('Energy (Wh)'); ax.set_title('HD-MHCS under different weight vectors'); save_fig(fig, 'F_pareto_weights')

## Scalability: tasks × VMs, wall-clock time and memory (same 4,000-evaluation budget)

In [ ]:
rows = []
for N in CFG['SCALE_TASKS']:
    for M in CFG['SCALE_VMS']:
        if N > len(TE): continue
        bt = {m: base_batch(TE[:N], M, m) for m in ['Min-Min', 'Max-Min', 'HHO']}; bt['HD-MHCS'] = hd_batch(TE[:N], M)
        for m in ['Min-Min', 'Max-Min', 'HHO', 'HD-MHCS']:
            for s in range(CFG['SCALE_SEEDS'] if m in ('HHO', 'HD-MHCS') else 1):
                r = run(m, bt[m], np.random.default_rng(4000 + s)); r.update(N=N, M=M, seed=s); rows.append(r)
        print(f'  scalability N={N} M={M} done')
SC_ = save_table(pd.DataFrame(rows).groupby(['N', 'M', 'method'])[['fitness_true', 'makespan', 'energy', 'sla', 'wall_s', 'peak_MB']].mean().reset_index(), 'T_scalability')
fig, ax = plt.subplots(figsize=(8, 5))
for m, g in SC_[SC_.M == CFG['SCALE_VMS'][0]].groupby('method'): ax.plot(g.N, g.wall_s, 'o-', label=m)
ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlabel('Tasks'); ax.set_ylabel('Wall-clock time (s)'); ax.legend(); save_fig(fig, 'F_scalability')

## Export

In [ ]:
with pd.ExcelWriter(os.path.join(OUT, 'HDMHCS_revision_results.xlsx')) as xw:
    for k, v in TABLES.items(): v.to_excel(xw, sheet_name=k[:31], index=False)
import shutil; shutil.make_archive(OUT, 'zip', OUT); print('Done:', OUT + '.zip')